# Chapter 11.5: Code Review Standards & PR Process

This notebook covers the code review process and PR standards for contributing
to vLLM and SGLang projects.

**Learning Objectives:**
- Understand vLLM and SGLang contribution guidelines
- Write effective PR descriptions
- Review code for performance, correctness, and style
- Handle review feedback professionally

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part11/chapter_11.5_code_review.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part11/chapter_11.5_code_review.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Contribution Guidelines Walkthrough

In [ ]:
# Demo: vLLM contribution guidelines overview

vllm_guidelines = """
vLLM Contribution Guidelines
============================

1. Before Contributing:
   - Check existing issues and PRs to avoid duplication
   - For large changes, open an RFC issue first
   - Join the vLLM Discord or Slack for discussion

2. Code Standards:
   - Follow PEP 8 (enforced by flake8/black)
   - Use type hints for all public APIs
   - Write docstrings for all public classes and methods
   - Maximum line length: 80 characters (code), 120 (comments)
   - Use black formatter with default settings
   - Use isort with black profile for imports

3. Testing Requirements:
   - All new features must have tests
   - Bug fixes should include a regression test
   - Tests must pass on CI before merge
   - Model correctness tests required for new model support

4. PR Process:
   - Fork the repo and create a feature branch
   - Write clear commit messages
   - Fill out the PR template completely
   - Request review from relevant maintainers
   - Address all review comments
   - Squash commits before merge (if requested)

5. Areas of Contribution:
   - New model support (most common)
   - Performance optimizations
   - Bug fixes
   - Documentation improvements
   - New features (discuss first)
"""

sglang_guidelines = """
SGLang Contribution Guidelines
==============================

1. Similar standards to vLLM, plus:
   - Focus on the language frontend and runtime separation
   - Triton kernel contributions welcome
   - Frontend (lang/) changes need backward compatibility
   - Runtime (srt/) changes need performance benchmarks

2. Key Differences from vLLM:
   - Smaller codebase, faster review cycles
   - More emphasis on Triton kernels vs CUDA C++
   - Language frontend has its own testing framework
   - Less formal RFC process for new features
"""

print(vllm_guidelines)
print(sglang_guidelines)

In [ ]:
# Demo: Understanding the vLLM PR template

pr_template = '''
## Summary / What does this PR do?
<!-- Describe the changes and their purpose -->

## Related Issues
<!-- Link to related issues: Fixes #123, Related to #456 -->

## Changes
<!-- Bulleted list of key changes -->
- 
- 

## Testing
<!-- How was this tested? -->
- [ ] Unit tests pass
- [ ] Integration tests pass  
- [ ] Benchmarks show no regression

## Performance Impact
<!-- If applicable, include benchmark results -->

## Breaking Changes
<!-- List any breaking changes -->
None

## Checklist
- [ ] Tests added/updated
- [ ] Documentation updated
- [ ] Type hints added
- [ ] No new warnings
- [ ] Lint/format checks pass
'''

print("vLLM PR Template:")
print("=" * 60)
print(pr_template)

## 2. Code Review Checklist

In [ ]:
# Demo: Comprehensive code review checklist

class CodeReviewChecklist:
    """Code review checklist for vLLM/SGLang PRs."""
    
    def __init__(self):
        self.categories = {
            "Correctness": [
                "Does the code do what the PR description says?",
                "Are edge cases handled (empty inputs, max values, errors)?",
                "Are there any race conditions in concurrent code?",
                "Are tensor shapes validated before kernel launches?",
                "Is memory properly freed (no leaks)?",
                "Are error messages helpful and specific?",
            ],
            "Performance": [
                "Are there unnecessary copies (CPU <-> GPU, .contiguous())?",
                "Are CUDA kernels launched with appropriate grid/block sizes?",
                "Could this cause GPU idle time (synchronous operations)?",
                "Is memory allocation in the hot path avoided?",
                "Are there Python loops that could be vectorized?",
                "Does this change affect CUDA graph compatibility?",
            ],
            "Code Quality": [
                "Is the code readable without excessive comments?",
                "Are variable names descriptive?",
                "Is the code DRY (no unnecessary duplication)?",
                "Are functions/methods at the right level of abstraction?",
                "Is the code consistent with the existing codebase style?",
            ],
            "Testing": [
                "Are there sufficient unit tests?",
                "Do tests cover edge cases and error paths?",
                "Are tests deterministic (no flakiness)?",
                "Do model tests compare against a reference implementation?",
                "Is test cleanup proper (no leaked resources)?",
            ],
            "Documentation": [
                "Are public APIs documented with docstrings?",
                "Are type hints present and correct?",
                "Are complex algorithms explained with comments?",
                "Is the PR description clear and complete?",
            ],
            "Security": [
                "No hardcoded credentials or tokens?",
                "User input is validated/sanitized?",
                "No arbitrary code execution paths?",
            ],
        }
    
    def print_checklist(self):
        print("Code Review Checklist for vLLM/SGLang")
        print("=" * 60)
        for category, items in self.categories.items():
            print(f"\n{category}:")
            print("-" * 40)
            for item in items:
                print(f"  [ ] {item}")
    
    def interactive_review(self, results=None):
        """Score a PR against the checklist."""
        if results is None:
            # Use default passing results for demo
            results = {}
            for cat in self.categories:
                results[cat] = [True] * len(self.categories[cat])
        
        print("\nReview Results:")
        print("=" * 60)
        total_checks = 0
        total_pass = 0
        
        for cat, items in self.categories.items():
            cat_results = results.get(cat, [True] * len(items))
            passed = sum(cat_results)
            total = len(items)
            total_checks += total
            total_pass += passed
            status = "PASS" if passed == total else "NEEDS WORK"
            print(f"  {cat:20s}: {passed}/{total} ({status})")
        
        print(f"\n  Overall: {total_pass}/{total_checks} "
              f"({100*total_pass/total_checks:.0f}%)")
        
        if total_pass == total_checks:
            print("  Recommendation: APPROVE")
        elif total_pass >= total_checks * 0.8:
            print("  Recommendation: APPROVE with minor comments")
        else:
            print("  Recommendation: REQUEST CHANGES")

checklist = CodeReviewChecklist()
checklist.print_checklist()

## 3. Walking Through a Real PR

In [ ]:
# Demo: Walk through a sample merged PR (simulated)

sample_pr = {
    "title": "feat: Add support for Mixtral MoE model",
    "number": 2450,
    "author": "contributor123",
    "status": "merged",
    "files_changed": 12,
    "additions": 850,
    "deletions": 45,
    "description": """
## Summary
This PR adds support for the Mixtral 8x7B Mixture of Experts model.

## Changes
- Added MixtralForCausalLM model implementation
- Implemented expert-parallel routing
- Added TopK gating with load balancing
- Added model correctness tests comparing with HF transformers

## Testing
- Model correctness test: greedy decoding matches HF within fp16 tolerance
- Throughput benchmark: 45 tokens/s on A100 (single GPU)
- Memory usage: 26GB for 8x7B model

## Performance
| Metric | Value |
|--------|-------|
| Throughput (tok/s) | 45.2 |
| TTFT (ms) | 180 |
| Memory (GB) | 26.3 |
""",
    "files": [
        {"path": "vllm/model_executor/models/mixtral.py", "additions": 450, "deletions": 0},
        {"path": "vllm/model_executor/models/__init__.py", "additions": 2, "deletions": 0},
        {"path": "vllm/model_executor/layers/moe.py", "additions": 200, "deletions": 0},
        {"path": "tests/models/test_mixtral.py", "additions": 120, "deletions": 0},
        {"path": "vllm/config.py", "additions": 15, "deletions": 5},
        {"path": "vllm/transformers_utils/config.py", "additions": 8, "deletions": 2},
    ],
    "review_comments": [
        {
            "reviewer": "maintainer_A",
            "file": "vllm/model_executor/layers/moe.py",
            "line": 45,
            "comment": "This expert routing could be more efficient. Consider using "
                       "torch.topk instead of sorting all experts.",
            "type": "performance",
            "resolved": True
        },
        {
            "reviewer": "maintainer_B",
            "file": "vllm/model_executor/models/mixtral.py",
            "line": 120,
            "comment": "Missing type hints for the forward method parameters.",
            "type": "style",
            "resolved": True
        },
        {
            "reviewer": "maintainer_A",
            "file": "tests/models/test_mixtral.py",
            "line": 30,
            "comment": "Please add a test for the case where num_experts_per_token > 1. "
                       "The routing logic differs in this case.",
            "type": "testing",
            "resolved": True
        },
        {
            "reviewer": "maintainer_C",
            "file": "vllm/model_executor/models/mixtral.py",
            "line": 85,
            "comment": "Potential issue: the expert weights should be gathered before "
                       "the all-reduce in tensor parallel mode. Currently, this would "
                       "give wrong results with TP > 1.",
            "type": "correctness",
            "resolved": True
        },
    ]
}

print(f"PR #{sample_pr['number']}: {sample_pr['title']}")
print(f"Author: {sample_pr['author']}")
print(f"Status: {sample_pr['status']}")
print(f"Changes: +{sample_pr['additions']} -{sample_pr['deletions']} in {sample_pr['files_changed']} files")
print("\n" + sample_pr['description'])

print("\nFiles Changed:")
print("-" * 60)
for f in sample_pr['files']:
    print(f"  {f['path']:50s} +{f['additions']} -{f['deletions']}")

print("\nReview Comments:")
print("-" * 60)
for i, c in enumerate(sample_pr['review_comments'], 1):
    resolved = "(resolved)" if c['resolved'] else "(pending)"
    print(f"\n  Comment {i} [{c['type']}] {resolved}")
    print(f"  Reviewer: {c['reviewer']}")
    print(f"  File: {c['file']}:{c['line']}")
    print(f"  {c['comment']}")

In [ ]:
# Demo: Analyzing the PR for review quality

def analyze_pr_quality(pr):
    """Analyze the quality of a PR."""
    scores = {}
    
    # Description quality
    desc = pr['description']
    desc_score = 0
    if '## Summary' in desc: desc_score += 1
    if '## Changes' in desc: desc_score += 1
    if '## Testing' in desc: desc_score += 1
    if '## Performance' in desc: desc_score += 1
    if any(word in desc.lower() for word in ['benchmark', 'throughput', 'latency']): 
        desc_score += 1
    scores['Description'] = desc_score / 5
    
    # Test coverage
    test_files = [f for f in pr['files'] if 'test' in f['path']]
    code_files = [f for f in pr['files'] if 'test' not in f['path']]
    test_lines = sum(f['additions'] for f in test_files)
    code_lines = sum(f['additions'] for f in code_files)
    test_ratio = test_lines / code_lines if code_lines > 0 else 0
    scores['Test Coverage'] = min(test_ratio / 0.3, 1.0)  # 30% test ratio = perfect
    
    # Review engagement
    comments = pr['review_comments']
    resolved = sum(1 for c in comments if c['resolved'])
    scores['Review Resolution'] = resolved / len(comments) if comments else 1.0
    
    # Comment types
    types = set(c['type'] for c in comments)
    scores['Review Thoroughness'] = len(types) / 4  # 4 types: correctness, performance, style, testing
    
    # PR size
    total_changes = pr['additions'] + pr['deletions']
    if total_changes < 400:
        scores['PR Size'] = 1.0  # Good: focused PR
    elif total_changes < 1000:
        scores['PR Size'] = 0.7  # OK: could be split
    else:
        scores['PR Size'] = 0.3  # Too large
    
    print("PR Quality Analysis")
    print("=" * 50)
    total_score = 0
    for category, score in scores.items():
        bar = '#' * int(score * 20)
        print(f"  {category:25s}: {score:.0%} {bar}")
        total_score += score
    
    avg = total_score / len(scores)
    print(f"\n  Overall Quality: {avg:.0%}")
    
    if avg >= 0.8:
        print("  Assessment: Excellent PR!")
    elif avg >= 0.6:
        print("  Assessment: Good PR with room for improvement.")
    else:
        print("  Assessment: Needs significant improvement.")

analyze_pr_quality(sample_pr)

## 4. Common Review Feedback Patterns

In [ ]:
# Demo: Common review feedback and how to address them

feedback_patterns = [
    {
        "pattern": "Missing type hints",
        "example_comment": "Please add type hints to the function signature.",
        "bad_code": '''def process_batch(batch, config):
    results = []
    for item in batch:
        result = self.model(item)
        results.append(result)
    return results''',
        "good_code": '''def process_batch(
    self,
    batch: List[torch.Tensor],
    config: SamplingParams,
) -> List[CompletionOutput]:
    results: List[CompletionOutput] = []
    for item in batch:
        result = self.model(item)
        results.append(result)
    return results'''
    },
    {
        "pattern": "Unnecessary GPU synchronization",
        "example_comment": "This torch.cuda.synchronize() blocks the CPU. Is it needed?",
        "bad_code": '''def forward(self, x):
    output = self.layer1(x)
    torch.cuda.synchronize()  # Unnecessary!
    output = self.layer2(output)
    torch.cuda.synchronize()  # Unnecessary!
    return output''',
        "good_code": '''def forward(self, x: torch.Tensor) -> torch.Tensor:
    output = self.layer1(x)
    output = self.layer2(output)
    return output
    # Synchronization only when needed (e.g., timing, debugging)'''
    },
    {
        "pattern": "Missing error handling",
        "example_comment": "What happens if the model name is invalid? Need validation.",
        "bad_code": '''def load_model(model_name):
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return model''',
        "good_code": '''def load_model(model_name: str) -> PreTrainedModel:
    if not model_name:
        raise ValueError("model_name cannot be empty")
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name)
    except OSError as e:
        raise ValueError(
            f"Failed to load model '{model_name}'. "
            f"Check that the model exists on HuggingFace Hub."
        ) from e
    return model'''
    },
    {
        "pattern": "Magic numbers",
        "example_comment": "What does 2048 represent? Please use a named constant.",
        "bad_code": '''def allocate_cache(self):
    self.cache = torch.zeros(32, 2048, 12, 64)
    self.max_blocks = 2048 // 16''',
        "good_code": '''# Constants
MAX_SEQ_LEN = 2048
NUM_HEADS = 12
HEAD_DIM = 64
BLOCK_SIZE = 16
NUM_LAYERS = 32

def allocate_cache(self):
    self.cache = torch.zeros(
        NUM_LAYERS, MAX_SEQ_LEN, NUM_HEADS, HEAD_DIM
    )
    self.max_blocks = MAX_SEQ_LEN // BLOCK_SIZE'''
    },
]

print("Common Review Feedback Patterns")
print("=" * 70)

for i, fb in enumerate(feedback_patterns, 1):
    print(f"\n--- Pattern {i}: {fb['pattern']} ---")
    print(f"Reviewer says: \"{fb['example_comment']}\"")
    print(f"\nBefore:")
    print(fb['bad_code'])
    print(f"\nAfter:")
    print(fb['good_code'])

## 5. How to Respond to Review Comments

In [ ]:
# Demo: Professional responses to review comments

response_examples = [
    {
        "reviewer_comment": "This code path has O(n^2) complexity. Can it be optimized?",
        "bad_response": "It's fine. n is always small.",
        "good_response": (
            "Good catch! I profiled this with n up to 1000 and it's not a "
            "bottleneck currently (< 1ms). However, I agree it's worth optimizing "
            "for future-proofing. I've updated it to use a hash-based approach "
            "which is O(n). See commit abc123."
        ),
    },
    {
        "reviewer_comment": "I think we should handle the case where the model config doesn't have this field.",
        "bad_response": "All models have this field.",
        "good_response": (
            "You're right. I checked and older model configs (before transformers 4.30) "
            "may not have this field. I've added a getattr with a sensible default:\n"
            "`num_experts = getattr(config, 'num_local_experts', 1)`\n"
            "This maintains backward compatibility."
        ),
    },
    {
        "reviewer_comment": "This approach seems overly complex. Can we simplify?",
        "bad_response": "It has to be this way.",
        "good_response": (
            "I agree the current implementation is complex. The reason for the "
            "current design is to handle [specific edge case]. I've refactored "
            "to extract the edge case handling into a separate method, which "
            "makes the main flow much clearer. I also added comments explaining "
            "the edge case. WDYT?"
        ),
    },
    {
        "reviewer_comment": "I disagree with this design choice. I think we should use X instead of Y.",
        "bad_response": "I prefer Y.",
        "good_response": (
            "Thanks for the suggestion! I considered X initially but chose Y because:\n"
            "1. Y has O(1) lookup vs O(n) for X in our use case\n"
            "2. Y is already used in the scheduler module (consistency)\n"
            "3. Y makes it easier to add [future feature]\n\n"
            "That said, I'm open to X if you think the tradeoffs favor it. "
            "Happy to discuss further!"
        ),
    },
]

print("How to Respond to Review Comments")
print("=" * 70)

for i, ex in enumerate(response_examples, 1):
    print(f"\n--- Example {i} ---")
    print(f"Reviewer: \"{ex['reviewer_comment']}\"")
    print(f"\n  Bad response:  \"{ex['bad_response']}\"")
    print(f"  Good response: \"{ex['good_response']}\"")

## 6. Code Review Automation Tools

In [ ]:
# Demo: Automated code review checks
import re

class AutomatedCodeReviewer:
    """Automated checks that can catch common issues before human review."""
    
    def __init__(self):
        self.issues = []
    
    def check_type_hints(self, code: str, filename: str):
        """Check for functions missing type hints."""
        pattern = r'def\s+(\w+)\s*\(([^)]*?)\)\s*:'
        for match in re.finditer(pattern, code):
            func_name = match.group(1)
            params = match.group(2)
            if func_name.startswith('_'):
                continue
            # Check for return type
            full_match = match.group(0)
            if '->' not in code[match.start():match.end()+20]:
                self.issues.append({
                    'file': filename,
                    'type': 'type_hint',
                    'severity': 'warning',
                    'message': f'Function "{func_name}" missing return type hint'
                })
    
    def check_cuda_sync(self, code: str, filename: str):
        """Check for unnecessary CUDA synchronization."""
        if 'torch.cuda.synchronize()' in code:
            lines = code.split('\n')
            for i, line in enumerate(lines):
                if 'torch.cuda.synchronize()' in line:
                    if '#' not in line or 'needed' not in line.lower():
                        self.issues.append({
                            'file': filename,
                            'type': 'performance',
                            'severity': 'warning',
                            'message': f'Line {i+1}: CUDA synchronize may block CPU. Add comment explaining why needed.'
                        })
    
    def check_magic_numbers(self, code: str, filename: str):
        """Check for magic numbers in code."""
        pattern = r'(?<!\w)(\d{3,})(?!\w)'
        for match in re.finditer(pattern, code):
            number = match.group(1)
            # Skip common acceptable numbers
            if number in ['1000', '1024', '2048', '4096', '8192']:
                continue
            # Check if it's a named constant assignment
            line_start = code.rfind('\n', 0, match.start()) + 1
            line = code[line_start:code.find('\n', match.end())]
            if '=' in line and line.strip()[0].isupper():
                continue  # It's a constant definition
            self.issues.append({
                'file': filename,
                'type': 'readability',
                'severity': 'info',
                'message': f'Number {number} should be a named constant'
            })
    
    def check_print_statements(self, code: str, filename: str):
        """Check for print statements (should use logging)."""
        if 'test' in filename:
            return  # prints OK in tests
        pattern = r'^\s*print\s*\('
        for i, line in enumerate(code.split('\n')):
            if re.match(pattern, line):
                self.issues.append({
                    'file': filename,
                    'type': 'logging',
                    'severity': 'warning',
                    'message': f'Line {i+1}: Use logger instead of print()'
                })
    
    def review(self, code: str, filename: str):
        self.issues = []
        self.check_type_hints(code, filename)
        self.check_cuda_sync(code, filename)
        self.check_magic_numbers(code, filename)
        self.check_print_statements(code, filename)
        return self.issues
    
    def print_report(self):
        print(f"\nAutomated Review: {len(self.issues)} issue(s) found")
        print("=" * 60)
        for issue in self.issues:
            icon = {'warning': 'WARN', 'info': 'INFO', 'error': 'ERROR'}[issue['severity']]
            print(f"  [{icon}] [{issue['type']}] {issue['file']}: {issue['message']}")

# Test the automated reviewer
sample_code = '''
import torch

class ModelRunner:
    def __init__(self, config):
        self.config = config
        self.max_batch = 256
    
    def forward(self, input_ids, positions):
        output = self.model(input_ids)
        torch.cuda.synchronize()
        print(f"Output shape: {output.shape}")
        result = self.process(output, 50272)
        return result
    
    def process(self, tensor, vocab_size):
        return tensor[:, -1, :vocab_size]
'''

reviewer = AutomatedCodeReviewer()
reviewer.review(sample_code, "vllm/model_runner.py")
reviewer.print_report()

---
## Hands-On Assignments

### Assignment 1: Review a Sample PR for Issues

**Objective:** Review the code below as if it were a PR and identify all issues.

In [ ]:
# Assignment 1: Review this code for issues

sample_pr_code = '''
# PR Title: "Add beam search support"
# PR Description: "Added beam search."

import torch
import numpy as np

class BeamSearchSampler:
    def __init__(self, beam_width, max_len, vocab_size, eos_token):
        self.beam_width = beam_width
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.eos = eos_token
        self.temperature = 1.0
    
    def search(self, model, input_ids):
        beams = [(input_ids, 0.0)]  # (sequence, score)
        completed = []
        
        for step in range(self.max_len):
            all_candidates = []
            for seq, score in beams:
                if seq[-1] == self.eos:
                    completed.append((seq, score))
                    continue
                
                logits = model(torch.tensor(seq).unsqueeze(0).cuda())
                torch.cuda.synchronize()
                probs = torch.softmax(logits[0, -1] / self.temperature, dim=-1)
                top_probs, top_ids = torch.topk(probs, self.beam_width)
                
                for i in range(self.beam_width):
                    new_seq = seq + [top_ids[i].item()]
                    new_score = score + np.log(top_probs[i].item())
                    all_candidates.append((new_seq, new_score))
            
            if not all_candidates:
                break
            
            all_candidates.sort(key=lambda x: x[1], reverse=True)
            beams = all_candidates[:self.beam_width]
            print(f"Step {step}: best score = {beams[0][1]:.4f}")
        
        completed.extend(beams)
        completed.sort(key=lambda x: x[1], reverse=True)
        return completed[0][0]
'''

print("PR Code to Review:")
print("=" * 60)
print(sample_pr_code)

print("\n" + "=" * 60)
print("TODO: Identify all issues in this PR code.")
print("")
print("Categories to check:")
print("  1. Correctness issues")
print("  2. Performance issues")
print("  3. Code quality / style issues")
print("  4. Missing tests")
print("  5. Documentation issues")
print("  6. PR description quality")
print("")
print("Write your review comments in the cell below.")

In [ ]:
# Assignment 1: Write your review here

your_review = {
    "pr_description_feedback": """
    TODO: Comment on the PR description quality
    """,
    "correctness_issues": [
        # TODO: List correctness issues you found
        # Example: "Line X: seq[-1] == self.eos check could fail if seq is empty"
    ],
    "performance_issues": [
        # TODO: List performance issues
    ],
    "style_issues": [
        # TODO: List code quality / style issues
    ],
    "missing_tests": [
        # TODO: What tests should be added?
    ],
    "overall_verdict": ""  # APPROVE / REQUEST_CHANGES / COMMENT
}

print("TODO: Complete your review above.")
print("Tip: A good review finds at least 5-8 issues across categories.")

### Assignment 2: Write a PR Description for a Feature

**Objective:** Write a complete, high-quality PR description for the feature described below.

In [ ]:
# Assignment 2: Write a PR description

feature_spec = """
Feature: Request Priority Queuing

You've implemented a feature that allows vLLM to prioritize certain
requests over others. The feature includes:

1. A new `priority` field in the API request (values: low, medium, high)
2. Modified scheduler to use priority-based ordering
3. Anti-starvation mechanism (aging) so low-priority requests
   eventually get served
4. Metrics for per-priority latency tracking

Files changed:
  - vllm/core/scheduler.py (modified scheduler logic)
  - vllm/entrypoints/openai/protocol.py (added priority field)
  - vllm/entrypoints/openai/api_server.py (pass priority through)
  - tests/core/test_priority_scheduler.py (new test file)
  - tests/entrypoints/test_priority_api.py (API tests)

Performance impact:
  - No throughput regression for single-priority workloads
  - High-priority requests see 30% lower TTFT under load
  - Low-priority requests see 20% higher TTFT under load
"""

print(feature_spec)
print("\n" + "=" * 60)
print("TODO: Write the PR title and description below.")
print("Use the vLLM PR template format.")

In [ ]:
# Assignment 2: Your PR description

pr_title = ""  # TODO: Write a concise PR title

pr_description = """
## Summary / What does this PR do?
<!-- TODO: Fill in -->

## Related Issues
<!-- TODO: Reference relevant issues -->

## Changes
<!-- TODO: Bulleted list of changes -->

## Testing
<!-- TODO: How was this tested? -->

## Performance Impact
<!-- TODO: Include benchmark results -->

## Breaking Changes
<!-- TODO: Any breaking changes? -->

## Checklist
- [ ] Tests added/updated
- [ ] Documentation updated
- [ ] Type hints added
- [ ] No new warnings
- [ ] Lint/format checks pass
"""

print(f"PR Title: {pr_title}")
print(pr_description)
print("TODO: Complete all sections of the PR description.")

### Assignment 3: Create a Code Review Checklist for a Module

**Objective:** Create a specialized code review checklist for the vLLM attention module.

In [ ]:
# Assignment 3: Create a module-specific review checklist

class ModuleReviewChecklist:
    """TODO: Create a review checklist specific to the attention module.
    
    The attention module (vllm/attention/) is critical for performance.
    Your checklist should cover:
    
    1. Memory safety (tensor shapes, bounds checking)
    2. Performance (kernel efficiency, memory access patterns)
    3. Correctness (numerical accuracy, edge cases)
    4. Compatibility (different GPU architectures, dtypes)
    5. Testing requirements specific to attention
    """
    
    def __init__(self, module_name="attention"):
        self.module_name = module_name
        self.checklist = {
            "Memory Safety": [
                # TODO: Add attention-specific memory safety checks
            ],
            "Performance": [
                # TODO: Add attention-specific performance checks
            ],
            "Correctness": [
                # TODO: Add correctness checks
            ],
            "GPU Compatibility": [
                # TODO: Add compatibility checks
            ],
            "Testing": [
                # TODO: Add testing requirements
            ],
        }
    
    def print_checklist(self):
        print(f"Code Review Checklist: {self.module_name} Module")
        print("=" * 60)
        for category, items in self.checklist.items():
            print(f"\n{category}:")
            if not items:
                print("  (TODO: Add items)")
            for item in items:
                print(f"  [ ] {item}")

checklist = ModuleReviewChecklist("attention")
checklist.print_checklist()
print("\nTODO: Fill in the checklist with at least 3 items per category.")

---
## Summary

In this notebook, we covered:

1. **Contribution guidelines** for vLLM and SGLang
2. **Code review checklist** across correctness, performance, quality, and testing
3. **PR walkthrough** analyzing a real-world contribution
4. **Common review feedback** patterns with before/after examples
5. **Professional responses** to review comments
6. **Automated code review** tools for catching common issues

**Next:** Chapter 11.6 - CI/CD Pipeline